In [1]:
from keys import openai_key, replicate_key, groq_key
from working import get_client, Debater, CollabDebate

In [2]:
from sam_dictionaries.sae import AutoEncoder

def load_sae(
    layer: int
):
    path = f"./sam_dictionaries/resid_out_layer{layer}/10_32768/ae.pt"

    return AutoEncoder.from_pretrained(path, device="cuda:0")

In [3]:
from working import ActivationCache
from nnsight import LanguageModel
from working.config import CacheConfig

layer = 3

# TODO
# 5_554
# 4_4365
# 3_11662
# 2_7440
# 1_7703
# 0_8313

model = LanguageModel("EleutherAI/pythia-70m-deduped", device_map="auto", dispatch=True)
sae = load_sae(
    layer=layer
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
cfg = CacheConfig

cache = ActivationCache(
    layer=layer,
    model=model,
    sae=sae,
    cfg = cfg
)

100%|██████████| 9/9 [00:31<00:00,  3.48s/it]

Activation Cache Size: torch.Size([1800, 128, 32768])


In [5]:
from working.sae import get_top_logits

feature_id = 8437 
examples_list, max_act = cache.get_top_examples(feature_id)

url = f"https://www.neuronpedia.org/api/feature/pythia-70m-deduped/{layer}-res-sm/{feature_id}"
top_logits = get_top_logits(url)

n_debaters = 3

debaters = [
    Debater(
        get_client("openai", openai_key),
        f"debater {id}"
    ) 
    for id in range(n_debaters)
]

debate = CollabDebate(debaters, examples_list, top_logits)

debate.run(max_rounds=2)

debate.history.save("openai", f"{layer}_{feature_id}")